# Phase 3 (A) — QAOA trên dữ liệu gốc, dùng Aer CPU

Dùng lại bốn JSON Phase 1–2 và cùng QUBO, gồm đầy đủ `x`, `y`, slack.
Bản này dùng **Aer SamplerV2**, mô phỏng statevector double trên CPU.
Mặc định thử lại `p0_reference` (23 qubit), `p=1`, 1024 shots, tối đa 35 lần đánh giá.

Chạy trong VS Code/Jupyter với kernel Conda `nt409-khiem`.
**Restart Kernel trước khi chạy** để giải phóng RAM và nạp module mới.
Chỉ lưu JSON cho lượt hoàn tất. Bước 5b đo một lượt lấy mẫu trước khi tối ưu toàn bộ.

## Bước 1 — Môi trường và đường dẫn

Aer và psutil đã có trong requirements-lock của môi trường hiện tại.
Nếu thiếu module, chạy trong terminal đã `conda activate nt409-khiem`:
`python -m pip install "qiskit-aer>=0.17,<0.18" "psutil>=7,<8"`, rồi Restart Kernel.
Không cần nâng cấp Python hoặc thay môi trường Conda.

In [2]:
import json
import sys
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations

import numpy as np
import pandas as pd
import qiskit
import qiskit_optimization
import qiskit_aer
import psutil

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "qubo.py").is_file() and (p / "data").is_dir()), None)
assert ROOT is not None, "Mở notebook trong repository nt409-task-placement."
assert (ROOT / "src" / "qaoa_experiment.py").is_file(), "Chép module mới vào src/."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.qubo import build_qubo
from src.ising import build_ising, ising_energy
from src.qaoa_experiment import (
    prepare_reference, probe_qaoa_resources, run_qaoa_experiment, save_experiment_report,
)

print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
print("Qiskit Optimization:", qiskit_optimization.__version__)
print("Project:", ROOT)
print("Aer:", qiskit_aer.__version__)
print("Available RAM (GiB):", round(psutil.virtual_memory().available / 2**30, 2))

Python: 3.12.14
Qiskit: 2.5.2
Qiskit Optimization: 0.7.0
Project: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Aer: 0.17.2
Available RAM (GiB): 16.85


## Bước 2 — Đọc lại bốn instance gốc

Không tạo dữ liệu mới. Tổng qubit gồm `x`, `y` và slack. Cột `one_statevector_MiB`
chỉ là kích thước **một mảng complex128** (`16 * 2**N` byte), không phải tổng
RAM của tiến trình mô phỏng. Bộ nhớ tạm và thời gian tối ưu còn phát sinh.

In [3]:
INSTANCE_IDS = ["p1_tight", "p1_choice", "p1_infeasible", "p0_reference"]
instances = {
    name: json.loads((ROOT / "data" / "instances" / f"{name}.json").read_text(encoding="utf-8"))
    for name in INSTANCE_IDS
}
assert all(instance["instance_id"] == name for name, instance in instances.items())
bundles = {name: build_qubo(instance) for name, instance in instances.items()}
ising = {name: build_ising(bundle) for name, bundle in bundles.items()}
size_rows = []
for name, bundle in bundles.items():
    operator, offset = ising[name]
    nbits = operator.num_qubits
    size_rows.append({
        "instance": name, "tasks": bundle["n"], "hosts": bundle["m"],
        "x+y": bundle["base_count"], "slack": nbits - bundle["base_count"],
        "qubits": nbits, "P": bundle["penalty"],
        "Pauli terms": len(operator), "offset": offset,
        "one_statevector_MiB": 16 * 2**nbits / 2**20,
    })
display(pd.DataFrame(size_rows))

,instance,tasks,hosts,x+y,slack,qubits,P,Pauli terms,offset,one_statevector_MiB
0,p1_tight,4,2,10,8,18,3.0,86,187.0,4.0
1,p1_choice,4,2,10,10,20,3.0,102,175.0,16.0
2,p1_infeasible,4,2,10,10,20,3.0,102,319.0,16.0
3,p0_reference,4,2,10,13,23,3.0,127,742.0,128.0


## Bước 3 — Kiểm tra QUBO → Ising

Kiểm tra vector 0, tất cả vector có một bit 1 và tất cả vector có hai bit 1.
Hai hàm đều bậc tối đa hai nên tập điểm này kiểm tra mọi hệ số.
Không cấp phát statevector 18–23 qubit ở bước này.

Số vector là `1 + N + N*(N-1)/2`. Notebook 04 kiểm tra thêm một nghiệm Phase 2;
notebook này sẽ kiểm tra nghiệm chính xác vừa tính ở Bước 4.

In [4]:
checked = {}
for name, bundle in bundles.items():
    operator, offset = ising[name]
    assert all(set(label) <= {"I", "Z"} and label.count("Z") <= 2
               for label in operator.paulis.to_labels())
    nbits = operator.num_qubits
    count = 0
    for active_count in (0, 1, 2):
        for indices in combinations(range(nbits), active_count):
            bits = np.zeros(nbits, dtype=int)
            bits[list(indices)] = 1
            expected = float(bundle["qubo"].objective.evaluate(bits))
            observed = ising_energy(operator, offset, bits)
            assert np.isclose(expected, observed, rtol=0, atol=1e-7), (name, bits)
            count += 1
    checked[name] = count
print("PASS — QUBO energy = Ising energy + offset:", checked)

PASS — QUBO energy = Ising energy + offset: {'p1_tight': 172, 'p1_choice': 211, 'p1_infeasible': 211, 'p0_reference': 277}


## Bước 4 — Đáp án chuẩn và mốc lấy mẫu đều chính xác

Đáp án gốc được tính từ các `x,y` hợp lệ, không cố định là 1 host.
`p1_infeasible` phải có `instance_feasible=False`, số host tối ưu là `None`,
nhưng QUBO vẫn có cực tiểu năng lượng hữu hạn (5 với dữ liệu hiện tại).

Mốc lấy mẫu đều vẫn xét **mọi chuỗi bit đầy đủ** với xác suất bằng nhau.
Ta tính chính xác mà không duyệt toàn bộ `2**N` chuỗi:

- Duyệt 1024 cách chọn `x,y` của mỗi fixture gốc.
- Tính số cách mã hóa từng giá trị slack; các trọng số trùng có thể tạo nhiều cách.
- Dùng trung bình và phương sai của slack để tính năng lượng trung bình.

Các mốc này là đáp án đối chứng, không được đưa vào phân phối mẫu QAOA.

In [5]:
references = {}
reference_rows = []
uniform_rows = []
for name, bundle in bundles.items():
    ref = prepare_reference(bundle)
    references[name] = ref
    operator, offset = ising[name]
    bits = ref["example_minimizer_bits"]
    assert np.isclose(ising_energy(operator, offset, bits), ref["minimum_energy"],
                      rtol=0, atol=1e-7)
    uniform = ref["uniform_baseline"]
    assert np.isclose(uniform["mean_qubo_energy"], offset, rtol=0, atol=1e-7)
    reference_rows.append({
        "instance": name, "instance_feasible": ref["instance_feasible"],
        "min QUBO E": ref["minimum_energy"],
        "best original hosts": ref["original_best_objective"],
        "ground-state full bitstrings": uniform["qubo_ground_state_bitstring_count"],
    })
    uniform_rows.append({
        "instance": name, "checked x+y": uniform["checked_base_vectors"],
        "total full bitstrings": uniform["total_bitstrings"],
        "feasible probability": uniform["feasible_probability"],
        "optimal placement probability": uniform["optimal_placement_probability"],
        "QUBO ground probability": uniform["ground_state_probability"],
        "mean QUBO E": uniform["mean_qubo_energy"],
    })
display(pd.DataFrame(reference_rows))
display(pd.DataFrame(uniform_rows))
print("PASS — exact reference bitstrings and uniform mean energies.")
print("None/NaN ở best original hosts của instance vô nghiệm = không có đáp án hợp lệ.")

,instance,instance_feasible,min QUBO E,best original hosts,ground-state full bitstrings
0,p1_tight,True,2.0,2.0,4
1,p1_choice,True,1.0,1.0,1
2,p1_infeasible,False,5.0,NaN,12
3,p0_reference,True,2.0,2.0,1


,instance,checked x+y,total full bitstrings,feasible probability,optimal placement probability,QUBO ground probability,mean QUBO E
0,p1_tight,1024,262144,0.003906,0.003906,1.525879e-05,187.0
1,p1_choice,1024,1048576,0.011719,0.000977,9.536743e-07,175.0
2,p1_infeasible,1024,1048576,0.000000,NaN,1.144409e-05,319.0
3,p0_reference,1024,8388608,0.000977,0.000977,1.192093e-07,742.0


PASS — exact reference bitstrings and uniform mean energies.
None/NaN ở best original hosts của instance vô nghiệm = không có đáp án hợp lệ.


## Bước 5 — Chọn lượt chạy

Giữ `p=1`, 1024 shots, tối đa 35 lần đánh giá và ngân sách 300 giây để kiểm tra
hiệu quả của thay đổi bộ mô phỏng. Chạy riêng `p0_reference` trước.
Sau đó có thể chọn `INSTANCE_IDS.copy()` để chạy bốn instance tuần tự.

Aer dùng tối đa 4 luồng CPU cho một mạch; không chạy song song nhiều mạch/shot.
`aer_memory_mb=2048` giới hạn bộ nhớ lưu trạng thái lượng tử của Aer,
**không phải giới hạn cứng toàn bộ RAM Python**. Kiểm tra còn ít nhất 2 GiB RAM
khả dụng trước mỗi lượt. Nếu thiếu, dừng để bạn giải phóng bộ nhớ.
`initial_point` theo thứ tự `[beta_0, ..., beta_(p-1), gamma_0, ..., gamma_(p-1)]`.

`max_seconds` vẫn được kiểm tra sau mỗi lần đánh giá, không ngắt giữa một lần
mô phỏng. Nếu dừng sớm, không lưu JSON kết quả hoặc đưa lượt đó vào manifest.

In [6]:
RUN_INSTANCE_IDS = ["p0_reference"]
# Chạy cả bốn tuần tự: RUN_INSTANCE_IDS = INSTANCE_IDS.copy()
CONFIG = {
    "seed": 42, "shots": 1024, "reps": 1, "maxiter": 35,
    "initial_point": [0.3, 0.7], "max_qubits": 23, "max_seconds": 300,
    "aer_threads": 4, "aer_memory_mb": 2048, "min_available_gib": 2.0,
}
assert RUN_INSTANCE_IDS and len(set(RUN_INSTANCE_IDS)) == len(RUN_INSTANCE_IDS)
assert set(RUN_INSTANCE_IDS) <= set(bundles), "Tên instance chưa được đọc ở Bước 2."
assert len(CONFIG["initial_point"]) == 2 * CONFIG["reps"]
assert all(bundles[name]["qubo"].get_num_vars() <= CONFIG["max_qubits"]
           for name in RUN_INSTANCE_IDS)
print("Selected:", RUN_INSTANCE_IDS)
print("Configuration:", CONFIG)

Selected: ['p0_reference']
Configuration: {'seed': 42, 'shots': 1024, 'reps': 1, 'maxiter': 35, 'initial_point': [0.3, 0.7], 'max_qubits': 23, 'max_seconds': 300, 'aer_threads': 4, 'aer_memory_mb': 2048, 'min_available_gib': 2.0}


## Bước 5b — Đo thử một lượt lấy mẫu

Chạy cell này rồi xem bảng trước khi chạy Bước 6. Đây là mạch tại góc khởi tạo,
không phải kết quả QAOA đã tối ưu và không được lưu vào thư mục kết quả.
`rough sampling total (s)` = thời gian lấy mẫu × (`maxiter` + 1), chỉ ước tính
phần mô phỏng; chưa gồm toàn bộ xử lý của optimizer/giải mã.

Nếu thời gian ước tính đã gần/vượt 300 giây hoặc RAM khả dụng rất thấp, hãy dừng
ở đây và gửi output; không chạy lặp lại vòng tối ưu nặng. Bản thân cell đo vẫn
chạy một mạch đầy đủ 23 qubit, với 1024 shots.


In [7]:
probes = {}
for name in RUN_INSTANCE_IDS:
    print("PROBE", name, flush=True)
    probes[name] = probe_qaoa_resources(bundles[name], CONFIG)
display(pd.DataFrame([{
    "instance": name, "qubits": p["num_qubits"],
    "setup (s)": p["setup_seconds"], "one sampling (s)": p["sampling_seconds"],
    "rough sampling total (s)": p["rough_maxiter_plus_final_sampling_seconds"],
    "Python peak RSS (MiB)": p["resources"]["process_peak_rss_mib"],
    "available RAM after probe (GiB)": p["resources"]["system_available_after_gib"],
    "shots": p["shots"],
} for name, p in probes.items()]))


PROBE p0_reference


,instance,qubits,setup (s),one sampling (s),rough sampling total (s),Python peak RSS (MiB),available RAM after probe (GiB),shots
0,p0_reference,23,0.094964,0.464576,16.724732,299.753906,16.914536,1024


## Bước 6 — Chạy tuần tự và chỉ lưu lượt hoàn tất

Mỗi instance dùng một QAOA mới. Mạch được transpile một lần trước vòng COBYLA.
Callback in lần đánh giá đầu và mỗi 5 lần tiếp theo.
`elapsed_seconds` là thời gian solver; thời gian giải mã được ghi riêng.
RAM đỉnh do **hệ điều hành ghi nhận trong toàn bộ vòng đời tiến trình Python**.
Nó bao gồm cả cell/lượt chạy trước đó; Restart Kernel để đo từ đầu.
Cách đo này ghi nhận cả cấp phát trong mã native của Aer, không cần polling thread.

Chỉ lượt `completed` có JSON trong `data/reference_results/phase3_runs/`.
`completed` nghĩa là solver kết thúc và có mẫu cuối, chưa khẳng định tìm được
cực tiểu toàn cục. Nếu lượt sau lỗi, kết quả hoàn tất của lượt trước vẫn còn.

In [8]:
reports = {}
report_paths = {}
for name in RUN_INSTANCE_IDS:
    print(f"START {name}: {ising[name][0].num_qubits} qubits — Aer CPU", flush=True)
    report = run_qaoa_experiment(bundles[name], CONFIG, references[name])
    if report["status"] != "completed":
        print(f"STOP {name}: {report['stop_reason']} — không lưu kết quả.", flush=True)
        continue
    path = save_experiment_report(ROOT, report)
    reports[name] = report
    report_paths[name] = path
    peak = report["resources"]["process_peak_rss_mib"]
    print(f"completed: {report['elapsed_seconds']:.2f}s; "
          f"Python peak RSS ≈ {peak:.1f} MiB; saved {path.name}", flush=True)

START p0_reference: 23 qubits — Aer CPU
  evaluation 1: mean E=793.263672, elapsed=0.6s
  evaluation 5: mean E=773.605469, elapsed=2.5s
  evaluation 10: mean E=707.354492, elapsed=5.0s
  evaluation 15: mean E=811.536133, elapsed=7.5s
  evaluation 20: mean E=696.104492, elapsed=9.9s
  evaluation 25: mean E=685.830078, elapsed=12.3s
completed: 14.70s; Python peak RSS ≈ 302.8 MiB; saved p0_reference_p1_seed42_20261003T090852250115Z.json


## Bước 7 — Đánh giá các mẫu thực sự quan sát được

- Hợp lệ: chỉ kiểm tra cách đặt task `x,y` theo Phase 1, chưa yêu cầu slack đúng.
- Placement tối ưu: hợp lệ và dùng số host bằng đáp án chuẩn riêng của instance.
- Ground state QUBO: cả `x,y,slack` đạt năng lượng QUBO nhỏ nhất.

Với `p1_infeasible`, tỷ lệ placement hợp lệ phải bằng 0; placement tối ưu là
`N/A` vì không tồn tại. Có thể đo được ground state QUBO nhưng mẫu đó vẫn vi phạm
bài toán gốc. Với instance khả thi, không đo được mẫu hợp lệ chưa chứng minh vô nghiệm.

Bảng top 10 được sắp theo tần suất, không theo chất lượng. `bitstring` dùng
cách in Qiskit (qubit 0 bên phải); các mẫu có cùng `x` có thể khác `y` hoặc slack.

In [9]:
def percent_or_na(value):
    return "N/A" if value is None else f"{100 * value:.6f}%"

for name, report in reports.items():
    print("\nINSTANCE:", name)
    if report["status"] != "completed":
        print("Status:", report["status"], "— chưa có mẫu cuối để đánh giá.")
        continue
    metrics = report["metrics"]
    uniform = report["reference"]["uniform_baseline"]
    display(pd.DataFrame([
        {"method": "Uniform (exact)",
         "feasible": percent_or_na(uniform["feasible_probability"]),
         "optimal placement": percent_or_na(uniform["optimal_placement_probability"]),
         "QUBO ground": percent_or_na(uniform["ground_state_probability"]),
         "mean QUBO E": uniform["mean_qubo_energy"]},
        {"method": "QAOA final samples",
         "feasible": percent_or_na(metrics["feasible_probability"]),
         "optimal placement": percent_or_na(metrics["optimal_placement_probability"]),
         "QUBO ground": percent_or_na(metrics["qubo_ground_state_probability"]),
         "mean QUBO E": metrics["final_sample_mean_qubo_energy"]},
    ]))
    print("Exact original optimum:", report["reference"]["original_best_objective"])
    print("Most frequent valid?:", metrics["most_frequent_valid"])
    print("Best observed valid placement:", metrics["best_observed_valid"])
    display(pd.DataFrame([{
        "probability": sample["probability"], "QUBO E": sample["energy"],
        "valid": sample["decoded"]["feasible"],
        "hosts": sample["decoded"]["objective"],
        "x": sample["decoded"]["x"], "y": sample["decoded"]["y"],
        "bitstring (Qiskit)": "".join(str(bit) for bit in reversed(sample["bits"])),
    } for sample in report["final_samples"][:10]]))


INSTANCE: p0_reference


,method,feasible,optimal placement,QUBO ground,mean QUBO E
0,Uniform (exact),0.097656%,0.097656%,0.000012%,742.000000
1,QAOA final samples,0.000000%,0.000000%,0.000000%,658.904297


Exact original optimum: 2
Most frequent valid?: False
Best observed valid placement: None


,probability,QUBO E,valid,hosts,x,y,bitstring (Qiskit)
0,0.000977,742.0,False,None,"[[1, 1], [0, 1], [0, 1], [1, 0]]","[1, 0]",10110100000010101101011
1,0.000977,952.0,False,None,"[[1, 1], [0, 0], [1, 1], [1, 0]]","[1, 0]",11010001011100101110011
2,0.000977,681.0,False,None,"[[0, 1], [0, 1], [1, 0], [0, 0]]","[0, 0]",00101110011100000011010
3,0.000977,493.0,False,None,"[[1, 1], [1, 1], [0, 1], [0, 0]]","[1, 0]",00000100001100100101111
4,0.000977,226.0,False,None,"[[1, 0], [0, 1], [0, 1], [0, 1]]","[0, 1]",10100010001011010101001
5,0.000977,909.0,False,None,"[[1, 1], [0, 0], [0, 0], [1, 0]]","[0, 0]",10011110011100001000011
6,0.000977,1281.0,False,None,"[[1, 1], [0, 1], [0, 0], [1, 0]]","[0, 0]",01001010111100001001011
7,0.000977,382.0,False,None,"[[0, 1], [0, 0], [0, 1], [1, 0]]","[0, 1]",01000110011111001100010
8,0.000977,1047.0,False,None,"[[1, 0], [0, 1], [1, 0], [0, 1]]","[0, 0]",11010010110010010011001
9,0.000977,1308.0,False,None,"[[0, 0], [0, 1], [0, 1], [0, 1]]","[0, 0]",01110110111100010101000


## Bước 8 — Tổng hợp các lượt đã hoàn tất

Manifest chỉ trỏ tới JSON của lượt hoàn tất. Nếu chưa có lượt hoàn tất,
không tạo manifest. `best_observed_valid=None` ở instance có nghiệm nghĩa là
QAOA chưa đo được nghiệm hợp lệ, không có nghĩa instance vô nghiệm.
Một seed và p=1 kiểm chứng luồng chạy, chưa đủ kết luận chất lượng thuật toán.

In [10]:
summary_rows = []
for name, report in reports.items():
    metrics = report["metrics"]
    best = None if metrics is None else metrics["best_observed_valid"]
    summary_rows.append({
        "instance": name, "qubits": report["num_qubits"], "status": report["status"],
        "seconds": report["elapsed_seconds"], "evals": report["cost_function_evals"],
        "Python peak RSS MiB": report["resources"]["process_peak_rss_mib"],
        "exact best hosts": report["reference"]["original_best_objective"],
        "observed best hosts": None if best is None else best["objective"],
        "feasible probability": None if metrics is None else metrics["feasible_probability"],
        "optimal placement probability": None if metrics is None else metrics["optimal_placement_probability"],
    })
display(pd.DataFrame(summary_rows))
if reports:
    manifest = {
        "schema_version": 1, "scope": "phase3_original_instances_single_seed_pilot",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "completed_instances": list(reports),
        "runs": [{"instance_id": name, "status": reports[name]["status"],
                  "report": report_paths[name].relative_to(ROOT).as_posix()}
                 for name in reports],
        "summary": summary_rows,
    }
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    manifest_path = ROOT / "data" / "reference_results" / "phase3_runs" / f"manifest_{stamp}.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, allow_nan=False)
                             + "\n", encoding="utf-8")
    print("Saved manifest:", manifest_path)
else:
    print("Chưa có lượt hoàn tất; không lưu manifest.")

,instance,qubits,status,seconds,evals,Python peak RSS MiB,exact best hosts,observed best hosts,feasible probability,optimal placement probability
0,p0_reference,23,completed,14.699607,29,302.824219,2,None,0,0


Saved manifest: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase3_runs\manifest_20261003T090933611435Z.json


## Sau lượt chạy

1. Lưu notebook và các JSON mới. Kiểm tra `status=completed`, thời gian và RAM đỉnh.
2. Giữ nguyên số liệu thực đo. Aer và StatevectorSampler có thể cho phân phối
   hữu hạn khác nhau dù cùng seed; không yêu cầu output QAOA giống từng số.
3. Có thể chạy các instance còn lại bằng cách sửa `RUN_INSTANCE_IDS`, chạy lại từ Bước 5.
4. Các JSON timeout cũ không bị tự động xóa. Nếu không cần giữ, xóa đúng báo cáo
   `p0_reference_p1_seed42_20261003T010712873163Z.json` và manifest chỉ chứa lượt đó
   `manifest_20261003T010713278064Z.json` trong `data/reference_results/phase3_runs/`.
5. B dùng đúng JSON gốc để đối chiếu heuristic/MILP; C nhận các báo cáo hoàn tất,
   phân biệt trường `method` để biết backend của từng lượt.

Không cần chạy lại notebook 00–04. Giảm RAM/thời gian mô phỏng không tự động
cải thiện xác suất nghiệm tối ưu của QAOA.